# Alternative RCM BA - 4. Verification of F2

**Alternative solution** (to the supervisor's downscaling pipeline) for the bias adjustment of the RCM:

1. **F1** (`GCM_BA/`, done): WDF + QDM of EC-Earth3 towards ERA5 on the EC-Earth3 grid (0.7 deg), applied to
   `gcm_2025_nohosing` -> the target climate, ERA5-like, at 0.7 deg.
2. **Reference quantiles at 0.25 deg** (BCSD principle, Wood et al. 2004), per calendar month m (the 3 months centred on
   m, as F1) and quantile: Q_ref = Q_ERA5,0.25 + I[Q_F1 - Q_ERA5,0.7] for tasmax, ratios (interpolated in log) for DTR
   and the pr wet days; pr: dry-day fraction first, p0_ref = p0_ERA5,0.25 + I[p0_F1 - p0_ERA5,0.7] (ISIMIP3BASD
   principle, Lange 2019). I = bilinear interpolation from 0.7 to 0.25 deg. The RCM does not enter the reference.
3. **F2**: QDM of `rcm_2025_nohosing` towards Q_ref (`bu.fit_F2`: the xsdba training with Q_ref as ref quantiles),
   applied unchanged to the four RCM runs with `bu.apply_F` (the code of F1), each run ranked in its own distribution.

The RCM keeps its own day-to-day sequence and spatial structure (they carry the AMOC effect): the correction acts on
the distribution of each pixel only.

1. **Calibration**: quantiles of `rcm_2025_nohosing` + F2 against the reference (per month window), raw against
   adjusted; mean absolute difference over European land of the domain.
2. **Change signal**: hosing - original (2025, 2050) of the seasonal means, raw RCM against RCM + F2: QDM ranks each
   run in its own distribution, so the change should be kept (slope of adjusted on raw change close to 1).
3. **Day-to-day structure kept** (answer to the objection on pixel-wise QM, Vrac & Friederichs 2015): lag-1
   autocorrelation of tasmax and pr, and probability of a wet day after a wet day, raw against adjusted. The
   comparison with ERA5 / CERRA of the same statistics (dependence diagnostics, VALUE framework, Maraun et al. 2015)
   is done for F1 and for the RCM on the historical climate.

In [ ]:
import os
os.environ.setdefault("HDF5_USE_FILE_LOCKING", "FALSE")   # BDAP/EOS refuses HDF5 file locks ("NetCDF: HDF error")
import sys, json, time, warnings
from pathlib import Path
for _p in reversed((Path.cwd(), *Path.cwd().parents)):   # ba_utils.py (repository root): this folder or any parent folder
    sys.path.insert(0, str(_p))
import numpy as np, pandas as pd, xarray as xr
import matplotlib.pyplot as plt
import ba_utils as bu
warnings.filterwarnings("ignore")

CONFIG_PATH = None   # None = config_<profile>.json found automatically (written by notebook 00)
USE_DASK_CLUSTER = False   # True = use several CPU cores (recommended on a JEO-lab / JEO-desk node)
cfg = bu.load_config(CONFIG_PATH)
CLIENT = bu.start_dask() if USE_DASK_CLUSTER else None
VARS = cfg["variables"]                       # pr, tasmax, tasmin (tasmin through DTR, as F1)

# ---- F1 (GCM_BA): the same tags as GCM_BA notebooks 2-3
CAL_PERIOD = ("1981", "2014")                 # calibration period of F1 (ERA5 / EC-Earth3 historical)
F1_TAG = f"F1_{CAL_PERIOD[0]}-{CAL_PERIOD[1]}_{bu.F1_VARIANT}_europe"
F1_PRODUCTS = bu.gcm_ba_dir(cfg, "GCM_bias_adjusted_0.7deg")

# ---- alternative solution
TARGET_GCM = "gcm_2025_nohosing"              # F1 product that gives the target climate of the RCM calibration
RCM_CAL = "rcm_2025_nohosing"                 # RCM run driven by it: F2 calibrated on it
RCM_RUNS = ["rcm_2025_nohosing", "rcm_2025_hosing", "rcm_2050_nohosing", "rcm_2050_hosing"]
MARGIN = 1.0                                  # degrees around the EFAS domain (LISFLOOD Europe)
REGION = {k: (bu.EFAS_BOX[k][0] - MARGIN, bu.EFAS_BOX[k][1] + MARGIN) for k in ("lat", "lon")}
COARSE_REGION = {k: (REGION[k][0] - 1.0, REGION[k][1] + 1.0) for k in ("lat", "lon")}   # + 1 coarse cell, for the interpolation
CHUNKS_SPACE = {"lat": 32, "lon": 64}
RCM_READY = bu.dataset_available(cfg, RCM_CAL, "pr")


def ref_years():
    """Years of the reference = years of the RCM calibration run (the F1 product over the same years); as long as the
    RCM is not in the config: all the years of the F1 product (to recompute once the RCM arrives)."""
    src = bu.read_raw(cfg, RCM_CAL, "pr") if RCM_READY else bu.read_processed(F1_PRODUCTS / f"{TARGET_GCM}_{F1_TAG}.nc", "pr")
    y = src.time.dt.year.values
    return str(y.min()), str(y.max())


REF_YEARS = ref_years()
QREF_TAG = f"{F1_TAG}_{TARGET_GCM}_{REF_YEARS[0]}-{REF_YEARS[1]}"
QREF_FILE = bu.alt_dir(cfg, "1_reference_quantiles") / f"qref_{QREF_TAG}.nc"
F2_DIR = bu.alt_dir(cfg, "2_F2_calculation", f"F2_{QREF_TAG}")
OUT = bu.alt_dir(cfg, "RCM_bias_adjusted_0.25deg")
print(f"region {REGION} | reference years {REF_YEARS[0]}-{REF_YEARS[1]}"
      + ("" if RCM_READY else " (RCM not in the config yet: all the years of the F1 product)"))
print("reference quantiles:", QREF_FILE, "\nF2:", F2_DIR, "\nproducts:", OUT)

In [ ]:
assert RCM_READY, f"{RCM_CAL}: RCM paths not in the config yet (notebook 00)"
qref = xr.open_dataset(QREF_FILE).load()
land = bu.keep_mask(qref) & bu.load_domain_mask(cfg, "era5", like=qref.q_tasmax.isel(month=0, quantiles=0))
raw = lambda name, v: bu.crop_bbox(bu.read_raw(cfg, name, v), REGION)
adj = lambda name: xr.open_dataset(OUT / f"{name}_F2_{QREF_TAG}.nc", chunks={"time": 31}, **bu._TIME_KW)
DONE = [n for n in RCM_RUNS if (OUT / f"{n}_F2_{QREF_TAG}.nc").exists()]
print("adjusted runs:", DONE)

## 1. Calibration: RCM 2025 no hosing + F2 against the reference quantiles

In [ ]:
nq = int(qref.attrs["nquantiles"])
rows = []
for k in ("tasmax", "dtr"):
    if f"q_{k}" not in qref:
        continue
    if k == "tasmax":
        r, a = raw(RCM_CAL, "tasmax"), adj(RCM_CAL)["tasmax"]
    else:
        r = (raw(RCM_CAL, "tasmax") - raw(RCM_CAL, "tasmin")).clip(min=bu.DTR_MIN)
        a = adj(RCM_CAL)["tasmax"] - adj(RCM_CAL)["tasmin"]
    r, a = (bu.subset_time(x, *REF_YEARS) for x in (r, a))
    for lab, x in (("raw", r), ("F2", a)):
        d = abs(bu.window_quantiles(x.assign_coords(lat=qref.lat, lon=qref.lon), nq, chunks_space=CHUNKS_SPACE).compute() - qref[f"q_{k}"])
        for tau in (0.1, 0.5, 0.9, 0.99):
            rows.append({"var": k, "quantile": tau, "series": lab, "mean |Q - Q_ref|": float(d.sel(quantiles=tau, method="nearest").where(land).mean())})
p0 = {lab: bu.window_dry_fraction(bu.subset_time(x, *REF_YEARS).assign_coords(lat=qref.lat, lon=qref.lon)).compute()
      for lab, x in (("raw", raw(RCM_CAL, "pr")), ("F2", adj(RCM_CAL)["pr"]))}
for lab, x in p0.items():
    rows.append({"var": "pr", "quantile": "dry-day fraction", "series": lab, "mean |Q - Q_ref|": float(abs(x - qref.p0_pr).where(land).mean())})
display(pd.DataFrame(rows).pivot_table(index=["var", "quantile"], columns="series", values="mean |Q - Q_ref|").round(3))

## 2. Change signal kept: hosing - original, raw against adjusted

In [ ]:
PAIRS = [("rcm_2025_hosing", "rcm_2025_nohosing"), ("rcm_2050_hosing", "rcm_2050_nohosing")]
rows = []
for h, o in [p for p in PAIRS if all(n in DONE for n in p)]:
    for v in ("tasmax", "tasmin", "pr"):
        for season in ("DJF", "JJA"):
            sm = lambda x: x.where(x.time.dt.season == season, drop=True).mean("time").compute()
            d_raw = sm(raw(h, v)) - sm(raw(o, v))
            d_adj = sm(adj(h)[v]) - sm(adj(o)[v])
            if v == "pr":
                d_raw, d_adj = (100 * d / sm(raw(o, v)) for d in (d_raw, d_adj))
            m = land.values & np.isfinite(d_raw.values) & np.isfinite(d_adj.values)
            x, y = d_raw.values[m], d_adj.values[m]
            ax, ay = x - x.mean(), y - y.mean()
            rows.append({"pair": f"{h} - {o}", "var": v + (" (%)" if v == "pr" else " (degC)"), "season": season,
                         "mean raw": x.mean(), "mean F2": y.mean(), "slope F2 on raw": (ax * ay).sum() / (ax * ax).sum(),
                         "corr": (ax * ay).sum() / np.sqrt((ax * ax).sum() * (ay * ay).sum())})
display(pd.DataFrame(rows).set_index(["pair", "var", "season"]).round(3))

## 3. Day-to-day structure kept: raw against adjusted (RCM 2025 no hosing)

In [ ]:
def lag1(x):
    a = x - x.mean("time")
    return (a * a.shift(time=1)).mean("time") / a.var("time")

def wet_after_wet(pr):
    w = pr >= bu.THRESH_PR
    return (w & w.shift(time=1, fill_value=False)).sum("time") / w.shift(time=1, fill_value=False).sum("time")

rows = []
for lab, src in (("raw", lambda v: raw(RCM_CAL, v)), ("F2", lambda v: adj(RCM_CAL)[v])):
    tx, pr = (bu.subset_time(src(v), *REF_YEARS).assign_coords(lat=qref.lat, lon=qref.lon) for v in ("tasmax", "pr"))
    for name, f in (("lag-1 autocorrelation tasmax", lag1(tx)), ("lag-1 autocorrelation pr", lag1(pr)),
                    ("P(wet | wet the day before)", wet_after_wet(pr))):
        rows.append({"statistic": name, "series": lab, "mean over land": float(f.where(land).mean().compute())})
display(pd.DataFrame(rows).pivot_table(index="statistic", columns="series", values="mean over land").round(3))